In [80]:
# ==============================
# Modell-Vergleich für Simap-Projekt
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# === 1. CSV laden ===
df = pd.read_csv(r"C:\Users\Raguj\PycharmProjects\MLOps-HS-25\data\raw\simap_last10d_sample.csv")

print(df.shape)
print(df.head())



# === 2. Textspalte bauen ===
df["text_attribute"] = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()

# === 3. Nur gültige Zeilen behalten ===
df = df.dropna(subset=["order_type", "award_value"])
df = df[df["order_type"] != "unknown"]




# Schritt : Sonderzeichen, Währungen und Apostrophe entfernen
df["award_value_clean"] = (
    df["award_value"].astype(str)
    .str.replace("CHF", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace("_", "", regex=False)
    .str.replace(",", ".", regex=False)  # Komma → Punkt für Dezimalzahlen
)

# Schritt : Ungültige Einträge in NaN umwandeln
df["award_value_clean"] = pd.to_numeric(df["award_value_clean"], errors="coerce")


bins=[0, 100_000, 1_000_000, 10_000_000, float("inf")]
labels=["klein", "mittel", "gross", "sehr gross"]

df["value_category"] = pd.cut(
    df["award_value_clean"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# NaN-Zeilen entfernen
df = df.dropna(subset=["order_type", "value_category"])


# === 4. X und y definieren ===
X = df[["text_attribute", "country", "canton", "process_type", "project_type"]].fillna("")

X = X.astype(str)


# Zielvariable
y_raw = {
    "order_type": df["order_type"],
    "value_category": df["value_category"]
}

targets = {}
label_encoders = {}
for target_name, y in y_raw.items():
    le = LabelEncoder()
    targets[target_name] = le.fit_transform(y)
    label_encoders[target_name] = le

# === 5. Vorverarbeitung ===
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=15000), "text_attribute"),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["country", "canton", "process_type", "project_type"]),
    ],
    remainder="drop"
)

# === 6. Modelle definieren ===
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC()
}

# === 7. Training für beide Zielvariablen ===
for target_name, y in targets.items():
    print(f"\n==============================")
    print(f"Training für Zielvariable: {target_name}")
    print(f"==============================")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.1, random_state=42, stratify=y
    )
    for name, clf in models.items():
        print(f"\n Modell: {name}")
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)


        print(classification_report(y_test, y_pred))


(2337, 24)
                             project_id                        publication_id  \
0  acdb20e1-2210-460e-add0-4b93be8f4445  f2a6c8d8-f7da-4ed3-9700-3dc0829b8e07   
1  24b03a8d-eaed-407f-9ff8-4d6ba3687a35  752c355e-7201-4b6b-865c-1210965db091   
2  9ef225df-8677-4ed3-9b27-f53a7ac04172  5b02f5a8-e937-400c-a222-c720065bef90   
3  e64990e8-c4cd-47c3-8338-6795d725647c  3fdb1772-4cd1-48b4-a762-55ea5e75444d   
4  0d814cfd-0e1e-44ee-b3e7-c6fb5b112337  84cbd94d-2893-4252-a1ac-03c513a92e9b   

  publication_date pub_type  \
0       2025-11-12    award   
1       2025-11-12    award   
2       2025-11-12    award   
3       2025-11-12    award   
4       2025-11-12    award   

                                               title  \
0  0731.110 - Strada Nazionale N24 - Comune di St...   
1  Construction d'une nouvelle aile de salles de ...   
2  CHENEVIERS III - Systèmes de nettoyage faiscea...   
3  Renouvellement des lits de soins intensifs et ...   
4                                  

In [3]:
import React, { useState, useEffect } from 'react'
import { Download } from 'lucide-react'
import Papa from 'papaparse'

const SIMAPClassifier = () => {
  const [projects, setProjects] = useState([]);
  const [labeledProjects, setLabeledProjects] = useState([]);
  const [currentIndex, setCurrentIndex] = useState(0);

  // Filter States
  const [filterKanton, setFilterKanton] = useState('');
  const [filterFirma, setFilterFirma] = useState('');
  const [filterMinBetrag, setFilterMinBetrag] = useState('');
  const [filterMaxBetrag, setFilterMaxBetrag] = useState('');
  const [filterTyp, setFilterTyp] = useState('');

  useEffect(() => {
    loadCSVData();
  }, []);

  const loadCSVData = async () => {
    try {
      const data = await window.fs.readFile('simap_projects.csv', { encoding: 'utf8' });
      Papa.parse(data, {
        header: true,
        dynamicTyping: true,
        skipEmptyLines: true,
        complete: (results) => {
          setProjects(results.data);
        }
      });
    } catch (error) {
      console.error('Error loading CSV:', error);
    }
  };

  const labelProject = (label) => {
    const currentProject = filteredProjects[currentIndex];
    if (!currentProject) return;

    const updated = labeledProjects.filter(p => p.project_id !== currentProject.project_id);
    setLabeledProjects([...updated, { ...currentProject, ist_interessant: label }]);

    if (currentIndex < filteredProjects.length - 1) {
      setCurrentIndex(currentIndex + 1);
    }
  };

  const exportTrainingData = () => {
    if (labeledProjects.length === 0) {
      alert('Keine gelabelten Projekte');
      return;
    }

    const csv = Papa.unparse(labeledProjects);
    const blob = new Blob([csv], { type: 'text/csv' });
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = 'simap_training_data.csv';
    a.click();
  };

  const filteredProjects = projects.filter(p => {
    if (filterKanton && p.canton !== filterKanton) return false;
    if (filterFirma && !p.contracting_authority?.toLowerCase().includes(filterFirma.toLowerCase())) return false;
    if (filterTyp && p.project_type !== filterTyp) return false;

    const betrag = p.award_amount || p.estimated_amount || 0;
    if (filterMinBetrag && betrag < parseFloat(filterMinBetrag)) return false;
    if (filterMaxBetrag && betrag > parseFloat(filterMaxBetrag)) return false;

    return true;
  });

  const currentProject = filteredProjects[currentIndex];
  const isLabeled = labeledProjects.find(p => p.project_id === currentProject?.project_id);

  const kantone = [...new Set(projects.map(p => p.canton).filter(Boolean))].sort();
  const typen = [...new Set(projects.map(p => p.project_type).filter(Boolean))].sort();

  return (
    <div className="min-h-screen bg-gray-50 p-4">
      <div className="max-w-5xl mx-auto">

        {/* Header */}
        <div className="bg-white rounded-lg shadow p-4 mb-4">
          <div className="flex justify-between items-center">
            <h1 className="text-2xl font-bold">SIMAP Klassifizierer</h1>
            <button
              onClick={exportTrainingData}
              disabled={labeledProjects.length === 0}
              className="flex items-center gap-2 px-4 py-2 bg-green-600 text-white rounded hover:bg-green-700 disabled:bg-gray-300"
            >
              <Download className="w-4 h-4" />
              Export ({labeledProjects.length})
            </button>
          </div>
          <div className="mt-2 text-sm text-gray-600">
            Projekt {currentIndex + 1} von {filteredProjects.length}
          </div>
        </div>

        {/* Filter */}
        <div className="bg-white rounded-lg shadow p-4 mb-4">
          <h2 className="font-bold mb-3">Filter</h2>
          <div className="grid grid-cols-2 md:grid-cols-3 gap-3">

            <div>
              <label className="block text-sm font-medium mb-1">Kanton</label>
              <select
                value={filterKanton}
                onChange={(e) => { setFilterKanton(e.target.value); setCurrentIndex(0); }}
                className="w-full border rounded px-2 py-1"
              >
                <option value="">Alle</option>
                {kantone.map(k => <option key={k} value={k}>{k}</option>)}
              </select>
            </div>

            <div>
              <label className="block text-sm font-medium mb-1">Projekt Typ</label>
              <select
                value={filterTyp}
                onChange={(e) => { setFilterTyp(e.target.value); setCurrentIndex(0); }}
                className="w-full border rounded px-2 py-1"
              >
                <option value="">Alle</option>
                {typen.map(t => <option key={t} value={t}>{t}</option>)}
              </select>
            </div>

            <div>
              <label className="block text-sm font-medium mb-1">Firma (enthält)</label>
              <input
                type="text"
                value={filterFirma}
                onChange={(e) => { setFilterFirma(e.target.value); setCurrentIndex(0); }}
                placeholder="z.B. Stadt Zürich"
                className="w-full border rounded px-2 py-1"
              />
            </div>

            <div>
              <label className="block text-sm font-medium mb-1">Min. Betrag (CHF)</label>
              <input
                type="number"
                value={filterMinBetrag}
                onChange={(e) => { setFilterMinBetrag(e.target.value); setCurrentIndex(0); }}
                placeholder="0"
                className="w-full border rounded px-2 py-1"
              />
            </div>

            <div>
              <label className="block text-sm font-medium mb-1">Max. Betrag (CHF)</label>
              <input
                type="number"
                value={filterMaxBetrag}
                onChange={(e) => { setFilterMaxBetrag(e.target.value); setCurrentIndex(0); }}
                placeholder="∞"
                className="w-full border rounded px-2 py-1"
              />
            </div>

            <div className="flex items-end">
              <button
                onClick={() => {
                  setFilterKanton('');
                  setFilterFirma('');
                  setFilterMinBetrag('');
                  setFilterMaxBetrag('');
                  setFilterTyp('');
                  setCurrentIndex(0);
                }}
                className="w-full px-3 py-1 bg-gray-200 rounded hover:bg-gray-300"
              >
                Filter zurücksetzen
              </button>
            </div>
          </div>
        </div>

        {/* Projekt Karte */}
        {currentProject ? (
          <div className="bg-white rounded-lg shadow p-4 mb-4">
            <h2 className="text-xl font-bold mb-3">{currentProject.title}</h2>

            {isLabeled && (
              <div className={`inline-block px-3 py-1 rounded mb-3 ${
                isLabeled.ist_interessant === 1 ? 'bg-green-100 text-green-800' : 'bg-red-100 text-red-800'
              }`}>
                {isLabeled.ist_interessant === 1 ? 'Interessant' : 'Nicht interessant'}
              </div>
            )}

            <div className="space-y-2 mb-4 text-sm">
              <div><strong>Beschreibung:</strong> {currentProject.description}</div>
              <div><strong>Auftraggeber:</strong> {currentProject.contracting_authority}</div>
              <div><strong>Ort:</strong> {currentProject.city}, {currentProject.canton}</div>
              <div><strong>Typ:</strong> {currentProject.project_type} / {currentProject.project_subtype}</div>
              <div><strong>Betrag:</strong> {
                currentProject.award_amount
                  ? `${currentProject.award_amount.toLocaleString()} ${currentProject.award_currency}`
                  : currentProject.estimated_amount
                    ? `ca. ${currentProject.estimated_amount.toLocaleString()} ${currentProject.estimated_currency}`
                    : 'Nicht angegeben'
              }</div>
              {currentProject.winner_name && (
                <div><strong>Gewinner:</strong> {currentProject.winner_name}</div>
              )}
            </div>

            <div className="flex gap-3">
              <button
                onClick={() => labelProject(1)}
                className="flex-1 py-2 bg-green-600 text-white rounded hover:bg-green-700 font-medium"
              >
                 Interessant
              </button>
              <button
                onClick={() => labelProject(0)}
                className="flex-1 py-2 bg-red-600 text-white rounded hover:bg-red-700 font-medium"
              >
                 Nicht interessant
              </button>
            </div>

            <div className="flex gap-3 mt-3">
              <button
                onClick={() => setCurrentIndex(Math.max(0, currentIndex - 1))}
                disabled={currentIndex === 0}
                className="px-4 py-2 bg-gray-200 rounded hover:bg-gray-300 disabled:opacity-50"
              >
                 Zurück
              </button>
              <button
                onClick={() => setCurrentIndex(Math.min(filteredProjects.length - 1, currentIndex + 1))}
                disabled={currentIndex === filteredProjects.length - 1}
                className="px-4 py-2 bg-gray-200 rounded hover:bg-gray-300 disabled:opacity-50"
              >
                Weiter
              </button>
            </div>
          </div>
        ) : (
          <div className="bg-white rounded-lg shadow p-12 text-center text-gray-600">
            Keine Projekte mit diesen Filtern
          </div>
        )}

      </div>
    </div>
  );
};

export default SIMAPClassifier;

SyntaxError: invalid syntax (2773712753.py, line 1)

In [4]:
import pandas as pd
import tkinter as tk
from tkinter import ttk, messagebox
from tkinter import filedialog


class SIMAPClassifier:
    def __init__(self, root):
        self.root = root
        self.root.title("SIMAP Klassifizierer")
        self.root.geometry("1000x800")

        # Daten
        self.csv_path = r"C:\Users\Raguj\PycharmProjects\MLOps-HS-25\data\simap_projects.csv"
        self.projects = pd.DataFrame()
        self.labeled_projects = []
        self.current_index = 0

        # Filter Variablen
        self.filter_kanton = tk.StringVar(value="")
        self.filter_firma = tk.StringVar(value="")
        self.filter_min_betrag = tk.StringVar(value="")
        self.filter_max_betrag = tk.StringVar(value="")
        self.filter_typ = tk.StringVar(value="")

        self.load_data()
        self.create_widgets()
        self.update_display()

    def load_data(self):
        try:
            self.projects = pd.read_csv(self.csv_path)
            print(f"CSV geladen: {len(self.projects)} Projekte")
        except Exception as e:
            messagebox.showerror("Fehler", f"CSV konnte nicht geladen werden: {e}")

    def create_widgets(self):
        # Header
        header_frame = tk.Frame(self.root, bg="white", padx=10, pady=10)
        header_frame.pack(fill=tk.X)

        tk.Label(header_frame, text="SIMAP Klassifizierer", font=("Arial", 20, "bold"), bg="white").pack(side=tk.LEFT)

        tk.Button(header_frame, text=f"Export ({len(self.labeled_projects)})",
                 command=self.export_data, bg="#10b981", fg="white", padx=20, pady=5).pack(side=tk.RIGHT)

        # Filter Frame
        filter_frame = tk.LabelFrame(self.root, text="Filter", padx=10, pady=10)
        filter_frame.pack(fill=tk.X, padx=10, pady=5)

        # Kanton
        tk.Label(filter_frame, text="Kanton:").grid(row=0, column=0, sticky=tk.W, pady=2)
        kantone = [""] + sorted(self.projects["canton"].dropna().unique().tolist())
        ttk.Combobox(filter_frame, textvariable=self.filter_kanton, values=kantone, width=20).grid(row=0, column=1, pady=2)

        # Typ
        tk.Label(filter_frame, text="Projekt Typ:").grid(row=0, column=2, sticky=tk.W, pady=2, padx=(20,0))
        typen = [""] + sorted(self.projects["project_type"].dropna().unique().tolist())
        ttk.Combobox(filter_frame, textvariable=self.filter_typ, values=typen, width=20).grid(row=0, column=3, pady=2)

        # Firma
        tk.Label(filter_frame, text="Firma (enthält):").grid(row=1, column=0, sticky=tk.W, pady=2)
        tk.Entry(filter_frame, textvariable=self.filter_firma, width=22).grid(row=1, column=1, pady=2)

        # Min Betrag
        tk.Label(filter_frame, text="Min. Betrag (CHF):").grid(row=1, column=2, sticky=tk.W, pady=2, padx=(20,0))
        tk.Entry(filter_frame, textvariable=self.filter_min_betrag, width=22).grid(row=1, column=3, pady=2)

        # Max Betrag
        tk.Label(filter_frame, text="Max. Betrag (CHF):").grid(row=2, column=0, sticky=tk.W, pady=2)
        tk.Entry(filter_frame, textvariable=self.filter_max_betrag, width=22).grid(row=2, column=1, pady=2)

        # Filter Buttons
        tk.Button(filter_frame, text="Filter anwenden", command=self.apply_filters,
                 bg="#3b82f6", fg="white", padx=10).grid(row=2, column=2, pady=5, padx=(20,5))
        tk.Button(filter_frame, text="Zurücksetzen", command=self.reset_filters,
                 bg="#6b7280", fg="white", padx=10).grid(row=2, column=3, pady=5)

        # Info Label
        self.info_label = tk.Label(self.root, text="", font=("Arial", 10))
        self.info_label.pack(pady=5)

        # Projekt Frame
        project_frame = tk.LabelFrame(self.root, text="Projekt Details", padx=10, pady=10)
        project_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)

        # Title
        self.title_label = tk.Label(project_frame, text="", font=("Arial", 14, "bold"), wraplength=900, justify=tk.LEFT)
        self.title_label.pack(anchor=tk.W, pady=(0,10))

        # Scrollable Text
        text_frame = tk.Frame(project_frame)
        text_frame.pack(fill=tk.BOTH, expand=True)

        scrollbar = tk.Scrollbar(text_frame)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        self.details_text = tk.Text(text_frame, height=15, wrap=tk.WORD, yscrollcommand=scrollbar.set)
        self.details_text.pack(fill=tk.BOTH, expand=True)
        scrollbar.config(command=self.details_text.yview)

        # Buttons Frame
        buttons_frame = tk.Frame(self.root, padx=10, pady=10)
        buttons_frame.pack(fill=tk.X)

        # Label Buttons
        tk.Button(buttons_frame, text="✓ Interessant", command=lambda: self.label_project(1),
                 bg="#10b981", fg="white", font=("Arial", 12, "bold"), padx=30, pady=10).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=5)
        tk.Button(buttons_frame, text="✗ Nicht interessant", command=lambda: self.label_project(0),
                 bg="#ef4444", fg="white", font=("Arial", 12, "bold"), padx=30, pady=10).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=5)

        # Navigation Frame
        nav_frame = tk.Frame(self.root, padx=10, pady=5)
        nav_frame.pack(fill=tk.X)

        tk.Button(nav_frame, text="← Zurück", command=self.prev_project, padx=20, pady=5).pack(side=tk.LEFT)
        tk.Button(nav_frame, text="Weiter →", command=self.next_project, padx=20, pady=5).pack(side=tk.RIGHT)

    def get_filtered_projects(self):
        df = self.projects.copy()

        if self.filter_kanton.get():
            df = df[df["canton"] == self.filter_kanton.get()]

        if self.filter_typ.get():
            df = df[df["project_type"] == self.filter_typ.get()]

        if self.filter_firma.get():
            df = df[df["contracting_authority"].str.contains(self.filter_firma.get(), case=False, na=False)]

        if self.filter_min_betrag.get():
            try:
                min_val = float(self.filter_min_betrag.get())
                df["betrag"] = df["award_amount"].fillna(df["estimated_amount"])
                df = df[df["betrag"] >= min_val]
            except ValueError:
                pass

        if self.filter_max_betrag.get():
            try:
                max_val = float(self.filter_max_betrag.get())
                df["betrag"] = df["award_amount"].fillna(df["estimated_amount"])
                df = df[df["betrag"] <= max_val]
            except ValueError:
                pass

        return df

    def apply_filters(self):
        self.current_index = 0
        self.update_display()

    def reset_filters(self):
        self.filter_kanton.set("")
        self.filter_firma.set("")
        self.filter_min_betrag.set("")
        self.filter_max_betrag.set("")
        self.filter_typ.set("")
        self.current_index = 0
        self.update_display()

    def update_display(self):
        filtered = self.get_filtered_projects()

        if filtered.empty:
            self.info_label.config(text="Keine Projekte mit diesen Filtern")
            self.title_label.config(text="")
            self.details_text.delete(1.0, tk.END)
            return

        if self.current_index >= len(filtered):
            self.current_index = len(filtered) - 1

        project = filtered.iloc[self.current_index]

        self.info_label.config(text=f"Projekt {self.current_index + 1} von {len(filtered)}")
        self.title_label.config(text=project["title"])

        # Check if already labeled
        is_labeled = any(p["project_id"] == project["project_id"] for p in self.labeled_projects)
        label_text = ""
        if is_labeled:
            label_val = next(p["ist_interessant"] for p in self.labeled_projects if p["project_id"] == project["project_id"])
            label_text = f"[{'INTERESSANT' if label_val == 1 else 'NICHT INTERESSANT'}]\n\n"

        details = f"{label_text}"
        details += f"Beschreibung: {project.get('description', 'N/A')}\n\n"
        details += f"Auftraggeber: {project.get('contracting_authority', 'N/A')}\n"
        details += f"Ort: {project.get('city', 'N/A')}, {project.get('canton', 'N/A')}\n"
        details += f"Typ: {project.get('project_type', 'N/A')} / {project.get('project_subtype', 'N/A')}\n"

        betrag = project.get('award_amount', project.get('estimated_amount', 'N/A'))
        currency = project.get('award_currency', project.get('estimated_currency', ''))
        if pd.notna(betrag):
            details += f"Betrag: {betrag:,.0f} {currency}\n"

        if pd.notna(project.get('winner_name')):
            details += f"Gewinner: {project.get('winner_name', '')}\n"

        self.details_text.delete(1.0, tk.END)
        self.details_text.insert(1.0, details)

    def label_project(self, label):
        filtered = self.get_filtered_projects()
        if filtered.empty:
            return

        project = filtered.iloc[self.current_index]

        # Remove old label if exists
        self.labeled_projects = [p for p in self.labeled_projects if p["project_id"] != project["project_id"]]

        # Add new label
        labeled_project = project.to_dict()
        labeled_project["ist_interessant"] = label
        self.labeled_projects.append(labeled_project)

        # Update export button
        for widget in self.root.winfo_children():
            if isinstance(widget, tk.Frame) and widget.cget("bg") == "white":
                for child in widget.winfo_children():
                    if isinstance(child, tk.Button) and "Export" in child.cget("text"):
                        child.config(text=f"Export ({len(self.labeled_projects)})")

        # Move to next
        if self.current_index < len(filtered) - 1:
            self.current_index += 1
            self.update_display()

    def next_project(self):
        filtered = self.get_filtered_projects()
        if self.current_index < len(filtered) - 1:
            self.current_index += 1
            self.update_display()

    def prev_project(self):
        if self.current_index > 0:
            self.current_index -= 1
            self.update_display()

    def export_data(self):
        if not self.labeled_projects:
            messagebox.showwarning("Warnung", "Keine gelabelten Projekte zum Exportieren")
            return

        file_path = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
            initialfile="simap_training_data.csv"
        )

        if file_path:
            df = pd.DataFrame(self.labeled_projects)
            df.to_csv(file_path, index=False)
            messagebox.showinfo("Erfolg", f"Daten exportiert: {len(self.labeled_projects)} Projekte")


if __name__ == "__main__":
    root = tk.Tk()
    app = SIMAPClassifier(root)
    root.mainloop()

CSV geladen: 100 Projekte
